In [ ]:
# cap_render.ipynb
"""Codi per renderitzar les capçaleres de les plantilles i comprovar si són correctes.
En fer display, desa la imatge renderitzada."""

from pathlib import Path
import tempfile
import atexit

import fitz
import win32com.client
from PIL import Image
from IPython.display import display, clear_output
import ipywidgets as widgets


# ============================================================
# SETTINGS
# ============================================================

ROOT_FOLDER = Path(
    r"C:\Users\55182183p\OneDrive - Generalitat de Catalunya\Obsidian_vaults\Gencat Energia\Bloc C - Plantilles\Plantilles\Plantilles úniques\Originals"
    #r"C:\Users\55182183p\OneDrive - Generalitat de Catalunya\Obsidian_vaults\Gencat Energia\Bloc C - Plantilles\Plantilla bona"
)
# Display the top 25% of the first page.
# Increase this to 0.35 or 0.40 if your headers are taller.
HEADER_HEIGHT_RATIO = 0.25

# Rendering resolution.
ZOOM = 2.0


# ============================================================
# FIND ALL DOCX FILES, INCLUDING IN SUBFOLDERS
# ============================================================

docx_files = sorted(
    file_path
    for file_path in ROOT_FOLDER.rglob("*.docx")
    if not file_path.name.startswith("~$")
)

if not ROOT_FOLDER.exists():
    raise FileNotFoundError(
        f"The folder does not exist:\n{ROOT_FOLDER}"
    )

if not docx_files:
    raise FileNotFoundError(
        f"No DOCX files were found in:\n{ROOT_FOLDER}"
    )

print(f"Found {len(docx_files)} DOCX files.")


# ============================================================
# START MICROSOFT WORD
# ============================================================

word = win32com.client.DispatchEx("Word.Application")
word.Visible = False
word.DisplayAlerts = 0

temporary_directory = tempfile.TemporaryDirectory()

current_index = 0
word_is_closed = False


def close_word_application():
    """Close the hidden Word application and remove temporary files."""

    global word_is_closed

    if word_is_closed:
        return

    try:
        word.Quit()
    except Exception:
        pass

    try:
        temporary_directory.cleanup()
    except Exception:
        pass

    word_is_closed = True


# Attempt to close Word when the notebook kernel exits.
atexit.register(close_word_application)


# ============================================================
# RENDER THE FIRST-PAGE HEADER
# ============================================================

def render_header(docx_path):
    """
    Open a DOCX file in Microsoft Word, export its first page to PDF,
    and return the top portion of the page as a PIL image.
    """

    pdf_path = (
        Path(temporary_directory.name)
        / f"document_{current_index}.pdf"
    )

    if pdf_path.exists():
        pdf_path.unlink()

    document = None

    try:
        document = word.Documents.Open(
            FileName=str(docx_path.resolve()),
            ConfirmConversions=False,
            ReadOnly=True,
            AddToRecentFiles=False,
            Visible=False,
            OpenAndRepair=False,
            NoEncodingDialog=True,
        )

        document.Repaginate()

        # Export only the first page.
        #
        # ExportFormat=17 means PDF.
        # Range=3 means export a specified page range.
        document.ExportAsFixedFormat(
            OutputFileName=str(pdf_path),
            ExportFormat=17,
            OpenAfterExport=False,
            OptimizeFor=0,
            Range=3,
            From=1,
            To=1,
            Item=0,
            IncludeDocProps=False,
            KeepIRM=True,
            CreateBookmarks=0,
            DocStructureTags=True,
            BitmapMissingFonts=True,
            UseISO19005_1=False,
        )

    finally:
        if document is not None:
            document.Close(SaveChanges=False)

    if not pdf_path.exists():
        raise RuntimeError(
            "Microsoft Word did not create the temporary PDF."
        )

    with fitz.open(str(pdf_path)) as pdf:
        page = pdf[0]
        page_rectangle = page.rect

        header_rectangle = fitz.Rect(
            page_rectangle.x0,
            page_rectangle.y0,
            page_rectangle.x1,
            page_rectangle.y0
            + page_rectangle.height * HEADER_HEIGHT_RATIO,
        )

        pixmap = page.get_pixmap(
            matrix=fitz.Matrix(ZOOM, ZOOM),
            clip=header_rectangle,
            alpha=False,
        )

        image = Image.frombytes(
            "RGB",
            (pixmap.width, pixmap.height),
            pixmap.samples,
        )

    return image


# ============================================================
# JUPYTER NOTEBOOK INTERFACE
# ============================================================

output = widgets.Output()

previous_button = widgets.Button(
    description="Previous",
    icon="arrow-left",
)

next_button = widgets.Button(
    description="Next",
    icon="arrow-right",
)

close_button = widgets.Button(
    description="Close Word",
    icon="stop",
    button_style="danger",
)

position_label = widgets.HTML()


def show_current_document():
    """Render and display the currently selected document."""

    docx_path = docx_files[current_index]

    previous_button.disabled = current_index == 0
    next_button.disabled = current_index == len(docx_files) - 1

    relative_path = docx_path.relative_to(ROOT_FOLDER)

    position_label.value = (
        f"<h3>Document {current_index + 1} "
        f"of {len(docx_files)}</h3>"
        f"<p><code>{relative_path}</code></p>"
    )

    with output:
        clear_output(wait=True)

        print("Rendering header...")

        try:
            header_image = render_header(docx_path)

            clear_output(wait=True)

            display(header_image)

            output_folder = Path("exported_headers")
            output_folder.mkdir(exist_ok=True)

            png_path = output_folder / f"{docx_path.stem}.png"
            header_image.save(png_path)


        except Exception as error:
            clear_output(wait=True)

            print("Unable to render this document:")
            print(docx_path)
            print()
            print(f"Error: {error}")


def show_previous_document(_):
    """Move to the previous document."""

    global current_index

    if current_index > 0:
        current_index -= 1
        show_current_document()


def show_next_document(_):
    """Move to the next document."""

    global current_index

    if current_index < len(docx_files) - 1:
        current_index += 1
        show_current_document()


def close_viewer(_):
    """Close Microsoft Word and disable the controls."""

    close_word_application()

    previous_button.disabled = True
    next_button.disabled = True
    close_button.disabled = True

    position_label.value = ""

    with output:
        clear_output()
        print("Microsoft Word has been closed.")


previous_button.on_click(show_previous_document)
next_button.on_click(show_next_document)
close_button.on_click(close_viewer)

controls = widgets.HBox(
    [
        previous_button,
        next_button,
        close_button,
    ]
)

display(position_label, controls, output)

show_current_document()

Found 23 DOCX files.


HTML(value='')

Output()